In [4]:
from pathlib import Path

import numpy as np
import scipy.optimize as spo

from cardiac_electrophysiology import builder
from cardiac_electrophysiology.utils import analysis, visualization

In [5]:
settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        vtu_mesh_path=Path("../data/patient_01/mesh_with_fibers_tags.vtu"),
        xdmf_mesh_path=Path("../data/patient_01/mesh.xdmf"),
        basis_vecs_path=Path("../data/patient_01/basis_vecs.npy"),
        log_file_path=Path("lsbip_logfile.log"),
        ground_truth_path=Path("ground_truth_from_prior.npy"),
        noisy_data_path=None,
        fiber_ensemble_path=None,
    ),
    strategies=builder.Strategies(
        ground_truth_strategy="from_file",
        mean_strategy="from_ground_truth",
        noisy_data_strategy="from_ground_truth",
    ),
    prior_parameters=builder.PriorParameters(
        kappa=0.01,
        tau=1,
        seed=0,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=3,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=100,
        noise_variance=1e-3,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

In [6]:
posterior_builder = builder.PosteriorBuilder(settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)

In [7]:
optimizer_options = {
    "disp": True,
    "maxiter": 1000,
    "ftol": 1e-3,
    "gtol": 1e-3,
    "maxls": 100,
}
initial_guess = np.zeros(additional_output.pv_mesh.number_of_cells)
map_estimate = spo.minimize(
    fun=posterior.evaluate_cost,
    jac=posterior.evaluate_gradient,
    x0=initial_guess,
    method="L-BFGS-B",
    options=optimizer_options,
)
print(map_estimate)
np.save("map_estimate.npy", map_estimate.x)

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 412.07639009709794
        x: [ 7.029e-02  2.196e-02 ...  1.389e-03 -1.691e-03]
      nit: 50
      jac: [ 1.383e-01  2.454e-02 ... -1.939e-02 -1.572e-02]
     nfev: 52
     njev: 52
 hess_inv: <31375x31375 LbfgsInvHessProduct with dtype=float64>


In [8]:
map_parameter = np.load("map_estimate.npy")
analysis_data = analysis.compute_map_result_analysis(
    map_parameter=map_parameter,
    posterior=posterior,
    additional_output=additional_output,
)

Prior mean angle L2-error: 57.91636737931716
Prior mean angle max-error: 0.8909772257385313
MAP angle L2-error: 55.50167134798393
MAP angle max-error: 1.5636409929152153
Prior mean predictive L2-error: 237.73358154296875
Prior mean predictive max-error: 5.31291389465332
MAP predictive L2-error: 86.67578887939453
MAP predictive max-error: 4.079843521118164


In [9]:
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.angle_prior_mean,
    clim=[0, np.pi / 2]
)
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.angle_ground_truth,
    clim=[0, np.pi / 2]
)
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.angle_map,
    clim=[0, np.pi / 2],
)

Widget(value='<iframe src="http://localhost:46767/index.html?ui=P_0x7f583dcca120_0&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:46767/index.html?ui=P_0x7f578845bc50_1&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:46767/index.html?ui=P_0x7f56f0453d90_2&reconnect=auto" class="pyvi…

In [10]:
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.diff_angle_truth_prior,
    clim=[0, np.pi/ 2],
    circular=False,
)
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.diff_angle_truth_map,
    clim=[0, np.pi / 2],
    circular=False,
)
visualization.visualize_arrival_times(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.diff_lat_truth_prior,
)
visualization.visualize_arrival_times(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.diff_lat_truth_map,
)

Widget(value='<iframe src="http://localhost:46767/index.html?ui=P_0x7f56f019f890_3&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:46767/index.html?ui=P_0x7f56f019f750_4&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:46767/index.html?ui=P_0x7f56e87e5590_5&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:46767/index.html?ui=P_0x7f56e87e6710_6&reconnect=auto" class="pyvi…